In [31]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("build_buckets") \
    .enableHiveSupport() \
    .getOrCreate()

26/04/27 23:06:32 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [32]:
spark.sql("show databases").show()

+--------------+
|     namespace|
+--------------+
|       default|
|      eco_data|
|       test_db|
|wsl_local_test|
+--------------+



In [33]:
spark.sql("USE eco_data")

print("=== 1. 数据倾斜检测（customer_id） ===")
spark.sql("""
SELECT 
    `Customer ID` as customer_id,
    COUNT(*) as record_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage
FROM fact_order_items
GROUP BY `Customer ID`
ORDER BY record_count DESC
LIMIT 15
""").show(truncate=False)

26/04/27 23:06:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/27 23:06:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/27 23:06:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/27 23:06:32 WARN ObjectStore: Falling back to ORM path due to direct SQL failure (this is not an error): class org.apache.derby.impl.jdbc.EmbedClob cannot be cast to class java.lang.String (org.apache.derby.impl.jdbc.EmbedClob is in unnamed module of loader 'app'; java.lang.String is in module java.base of loader 'bootstrap') at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:632) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectS

=== 1. 数据倾斜检测（customer_id） ===
+-----------+------------+----------+
|customer_id|record_count|percentage|
+-----------+------------+----------+
|85775      |2524        |0.43      |
|163        |2349        |0.40      |
|35         |1877        |0.32      |
|33         |1397        |0.24      |
|31025      |1369        |0.23      |
|806        |1310        |0.22      |
|1404       |1269        |0.22      |
|767        |1234        |0.21      |
|820        |1190        |0.20      |
|58         |1182        |0.20      |
|36         |1147        |0.20      |
|56         |1043        |0.18      |
|800        |1041        |0.18      |
|43         |1002        |0.17      |
|114        |826         |0.14      |
+-----------+------------+----------+



26/04/27 23:06:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/27 23:06:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/27 23:06:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/27 23:06:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [34]:
print("\n=== 2. 分区剪枝效果检查 ===")
spark.sql("EXPLAIN EXTENDED SELECT * FROM fact_order_items WHERE year = 2017 AND month = 3 LIMIT 10").show(truncate=False)


=== 2. 分区剪枝效果检查 ===
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|plan                                                                                                                                                                                                                                                                                                                                                |
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

26/04/27 23:06:33 WARN ObjectStore: Falling back to ORM path due to direct SQL failure (this is not an error): class org.apache.derby.impl.jdbc.EmbedClob cannot be cast to class java.lang.String (org.apache.derby.impl.jdbc.EmbedClob is in unnamed module of loader 'app'; java.lang.String is in module java.base of loader 'bootstrap') at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:632) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.loopJoinOrderedResult(MetaStoreDirectSql.java:955) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.getPartitionsFromPartitionIds(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.access$800(MetaStoreDirectSql.java:92) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$2.run(MetaStoreDirectSql.java:492) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.runBatched(MetaStoreDir

In [35]:
print("\n=== 3. 当前事实表分区信息 ===")
spark.sql("SHOW PARTITIONS fact_order_items").show(20)


=== 3. 当前事实表分区信息 ===
+------------------+
|         partition|
+------------------+
|year=2016/month=10|
|year=2016/month=11|
|year=2016/month=12|
| year=2016/month=7|
| year=2016/month=8|
| year=2016/month=9|
| year=2017/month=1|
|year=2017/month=10|
|year=2017/month=11|
|year=2017/month=12|
| year=2017/month=2|
| year=2017/month=3|
| year=2017/month=4|
| year=2017/month=5|
| year=2017/month=6|
| year=2017/month=7|
| year=2017/month=8|
| year=2017/month=9|
| year=2018/month=1|
| year=2018/month=2|
+------------------+
only showing top 20 rows



In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SkewFixWithSplit") \
    .master("local[*]") \
    .enableHiveSupport() \
    .getOrCreate()

# 执行加盐优化查询
print("=== 加盐打散优化后 customer_id 分布 ===")
spark.sql("""
WITH
skewed AS (
    SELECT customer_id
    FROM fact_order_items
    GROUP BY customer_id
    HAVING COUNT(*) > 1000
),
salted AS (
    SELECT
        CASE
            WHEN customer_id IN (SELECT customer_id FROM skewed)
            THEN CONCAT(customer_id, '_', CAST(RAND() * 10 AS INT))
            ELSE customer_id
        END AS salted_id
    FROM fact_order_items
),
partial AS (
    SELECT salted_id, COUNT(*) AS cnt
    FROM salted
    GROUP BY salted_id
)
SELECT
    SPLIT(salted_id, '_')[0] AS customer_id,
    SUM(cnt) AS record_count
FROM partial
GROUP BY SPLIT(salted_id, '_')[0]
ORDER BY record_count DESC
LIMIT 10
""").show(truncate=False)


=== 加盐打散优化后 customer_id 分布 ===


26/04/27 23:06:33 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `customer_id` cannot be resolved. Did you mean one of the following? [`Customer ID`, `item_id`, `increment_id`, `status`, `created_at`].; line 12 pos 17;
'WithCTE
:- 'CTERelationDef 21, false
:  +- 'SubqueryAlias skewed
:     +- 'UnresolvedHaving (count(1) > 1000)
:        +- 'Aggregate ['customer_id], ['customer_id]
:           +- SubqueryAlias spark_catalog.eco_data.fact_order_items
:              +- Relation spark_catalog.eco_data.fact_order_items[item_id#347,increment_id#348,created_at#349,price#350,qty_ordered#351,total_value#352,grand_total#353,discount_amount#354,status#355,payment_method#356,sku#357,Customer ID#358,category_name_1#359,year#360,month#361] parquet
:- 'CTERelationDef 22, false
:  +- 'SubqueryAlias salted
:     +- 'Project [CASE WHEN 'customer_id IN (list#653 []) THEN 'CONCAT('customer_id, _, cast((rand(-6436300704938836661) * cast(10 as double)) as int)) ELSE 'customer_